In [1]:
import sqlite3
import hashlib
from datetime import datetime

DB_NAME = "core_data.db"

def init_db():
    # เชื่อมต่อกับฐานข้อมูล (ถ้ายังไม่มีไฟล์ ระบบจะสร้างให้เอง)
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    # 1. ตาราง Users: เก็บข้อมูลสมาชิกและระดับสิทธิ์
    c.execute('''CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT UNIQUE NOT NULL,
        password_hash TEXT NOT NULL,
        email TEXT,
        role TEXT DEFAULT 'user', -- 'admin' หรือ 'user'
        created_at DATETIME
    )''')

    # 2. ตาราง Predictions (History): เก็บประวัติการตรวจข่าว
    # เพิ่มคอลัมน์ source_url สำหรับฟีเจอร์ดึงข่าวจากลิงก์
    c.execute('''CREATE TABLE IF NOT EXISTS predictions (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id INTEGER,
        news_title TEXT,
        news_text TEXT,
        source_url TEXT,         -- เก็บ URL ถ้าใช้ฟีเจอร์ Scrape
        result TEXT,             -- 'Real' หรือ 'Fake'
        confidence REAL,         -- ค่าความมั่นใจ (%)
        timestamp DATETIME,
        FOREIGN KEY (user_id) REFERENCES users(id)
    )''')

    # 3. ตาราง Feedbacks: เก็บการแจ้งผลผิดจาก User
    c.execute('''CREATE TABLE IF NOT EXISTS feedbacks (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        prediction_id INTEGER,    -- เชื่อมกับประวัติการทาย
        user_report TEXT,         -- 'Correct' หรือ 'Incorrect'
        comment TEXT,
        status TEXT DEFAULT 'pending', -- 'pending' หรือ 'reviewed'
        timestamp DATETIME,
        FOREIGN KEY (prediction_id) REFERENCES predictions(id)
    )''')

    # 4. ตาราง Trending_News: สำหรับ Admin จัดการข่าวเด่น
    c.execute('''CREATE TABLE IF NOT EXISTS trending_news (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        headline TEXT NOT NULL,
        content TEXT,
        label TEXT,               -- 'Fake', 'Warning', 'True'
        view_count INTEGER DEFAULT 0,
        updated_at DATETIME
    )''')

    # 5. ตาราง System_Logs: สำหรับทำ Analytics กราฟการใช้งาน
    c.execute('''CREATE TABLE IF NOT EXISTS system_logs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id INTEGER,
        action TEXT,              -- 'LOGIN', 'CHECK_NEWS', 'SCRAPE_URL'
        timestamp DATETIME,
        FOREIGN KEY (user_id) REFERENCES users(id)
    )''')

    # --- ส่วนการสร้าง Admin เริ่มต้น (Default Admin) ---
    # รหัสผ่านคือ 1234
    admin_username = "admin"
    admin_pass_hash = hashlib.sha256("1234".encode()).hexdigest()
    
    try:
        c.execute('''INSERT INTO users (username, password_hash, role, created_at) 
                     VALUES (?, ?, ?, ?)''', 
                  (admin_username, admin_pass_hash, 'admin', datetime.now()))
        print("✅ สร้าง Admin Account สำเร็จ (User: admin / Pass: 1234)")
    except sqlite3.IntegrityError:
        print("⚠️ มีบัญชี Admin อยู่แล้ว ข้ามการสร้างใหม่")

    conn.commit()
    conn.close()
    print(f"🚀 ฐานข้อมูล {DB_NAME} พร้อมใช้งานแล้ว!")

if __name__ == "__main__":
    init_db()

✅ สร้าง Admin Account สำเร็จ (User: admin / Pass: 1234)
🚀 ฐานข้อมูล core_data.db พร้อมใช้งานแล้ว!


C:\Users\tt_pe\AppData\Local\Temp\ipykernel_12596\1037397156.py:72: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  c.execute('''INSERT INTO users (username, password_hash, role, created_at)


In [2]:
import sqlite3
import hashlib
from datetime import datetime

DB_NAME = "core_data.db"

def get_connection():
    return sqlite3.connect(DB_NAME, check_same_thread=False)

# ==========================================
# 1. CRUD สำหรับ Users (สมัครสมาชิก/Login)
# ==========================================

def create_user(username, password, email, role='user'):
    conn = get_connection()
    c = conn.cursor()
    pw_hash = hashlib.sha256(password.encode()).hexdigest()
    try:
        c.execute("INSERT INTO users (username, password_hash, email, role, created_at) VALUES (?, ?, ?, ?, ?)",
                  (username, pw_hash, email, role, datetime.now()))
        conn.commit()
        return True
    except: return False
    finally: conn.close()

def authenticate_user(username, password):
    conn = get_connection()
    c = conn.cursor()
    pw_hash = hashlib.sha256(password.encode()).hexdigest()
    c.execute("SELECT id, username, role FROM users WHERE username = ? AND password_hash = ?", (username, pw_hash))
    user = c.fetchone()
    conn.close()
    return user # คืนค่า (id, username, role) ถ้าสำเร็จ

# ==========================================
# 2. CRUD สำหรับ Predictions (ประวัติการเช็คข่าว)
# ==========================================

def create_prediction(user_id, title, text, url, result, confidence):
    conn = get_connection()
    c = conn.cursor()
    c.execute("""INSERT INTO predictions (user_id, news_title, news_text, source_url, result, confidence, timestamp) 
                 VALUES (?, ?, ?, ?, ?, ?, ?)""",
              (user_id, title, text, url, result, confidence, datetime.now()))
    conn.commit()
    conn.close()

def read_user_history(user_id):
    conn = get_connection()
    c = conn.cursor()
    c.execute("SELECT timestamp, news_title, result, confidence FROM predictions WHERE user_id = ? ORDER BY timestamp DESC", (user_id,))
    data = c.fetchall()
    conn.close()
    return data

# ==========================================
# 3. CRUD สำหรับ Feedbacks (Admin รีวิวคำร้อง)
# ==========================================

def create_feedback(prediction_id, report_type, comment):
    conn = get_connection()
    c = conn.cursor()
    c.execute("INSERT INTO feedbacks (prediction_id, user_report, comment, timestamp) VALUES (?, ?, ?, ?)",
              (prediction_id, report_type, comment, datetime.now()))
    conn.commit()
    conn.close()

def read_all_feedbacks():
    conn = get_connection()
    c = conn.cursor()
    # Join ตารางเพื่อให้ Admin เห็นว่า Feedback นี้มาจากข่าวไหน
    c.execute("""SELECT f.id, p.news_title, f.user_report, f.comment, f.status 
                 FROM feedbacks f JOIN predictions p ON f.prediction_id = p.id""")
    data = c.fetchall()
    conn.close()
    return data

# ==========================================
# 4. CRUD สำหรับ Trending News (จัดการข่าวเด่น)
# ================================

def create_trending(headline, content, label):
    conn = get_connection()
    c = conn.cursor()
    c.execute("INSERT INTO trending_news (headline, content, label, updated_at) VALUES (?, ?, ?, ?)",
              (headline, content, label, datetime.now()))
    conn.commit()
    conn.close()

def update_trending(news_id, headline, content, label):
    conn = get_connection()
    c = conn.cursor()
    c.execute("UPDATE trending_news SET headline=?, content=?, label=?, updated_at=? WHERE id=?",
              (headline, content, label, datetime.now(), news_id))
    conn.commit()
    conn.close()

def delete_trending(news_id):
    conn = get_connection()
    c = conn.cursor()
    c.execute("DELETE FROM trending_news WHERE id=?", (news_id,))
    conn.commit()
    conn.close()

# ==========================================
# 5. CRUD สำหรับ Analytics (Logs)
# ==========================================

def create_log(user_id, action):
    conn = get_connection()
    c = conn.cursor()
    c.execute("INSERT INTO system_logs (user_id, action, timestamp) VALUES (?, ?, ?)",
              (user_id, action, datetime.now()))
    conn.commit()
    conn.close()

def read_logs_for_chart():
    conn = get_connection()
    c = conn.cursor()
    # ดึงเวลามาจัดกลุ่มเพื่อนทำกราฟ Activity by Time
    c.execute("SELECT strftime('%H', timestamp) as hour, COUNT(*) as count FROM system_logs GROUP BY hour")
    data = c.fetchall()
    conn.close()
    return data